<a href="https://colab.research.google.com/github/Sabareesh279/DataBootcampFinal/blob/final-final/Final_project_code1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

sns.set(style="whitegrid", context="notebook")
!pip install ucimlrepo


In [2]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
predict_students_dropout_and_academic_success = fetch_ucirepo(id=697)

# data (as dataframes)
X = predict_students_dropout_and_academic_success.data.features
y = predict_students_dropout_and_academic_success.data.targets


print(predict_students_dropout_and_academic_success.variables)



                                              name     role         type  \
0                                   Marital Status  Feature      Integer   
1                                 Application mode  Feature      Integer   
2                                Application order  Feature      Integer   
3                                           Course  Feature      Integer   
4                       Daytime/evening attendance  Feature      Integer   
5                           Previous qualification  Feature      Integer   
6                   Previous qualification (grade)  Feature   Continuous   
7                                      Nacionality  Feature      Integer   
8                           Mother's qualification  Feature      Integer   
9                           Father's qualification  Feature      Integer   
10                             Mother's occupation  Feature      Integer   
11                             Father's occupation  Feature      Integer   
12          

In [3]:
X_features = predict_students_dropout_and_academic_success.data.features
y_target = predict_students_dropout_and_academic_success.data.targets['Target'].to_frame()

#  Convert the target column to integer data (0 - dropout, 1 - enrolled, 2 - graduated)

category_order = [['Dropout', 'Enrolled', 'Graduate']]
ordinal_encoder = OrdinalEncoder(categories=category_order, handle_unknown='use_encoded_value', unknown_value=-1)

y_encoded = ordinal_encoder.fit_transform(y_target)
y_encoded = y_encoded.ravel() # Flatten to a 1D array for scikit-learn models

# ColumnTransformer to scale all numerical features
numerical_features = X_features.columns.tolist()

# Create the preprocessor for features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features)
    ],
    remainder='passthrough')




In [5]:
# Create a pipeline where all data is scaled and KNN is used
# We can use KNeighborsClassifier, now that Target is discrete (0, 1, 2).
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', KNeighborsClassifier(n_neighbors=19))])

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_features, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

# Fit the pipeline to the training data
pipeline.fit(X_train, y_train)

# Make predictions on the test set
y_pred = pipeline.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)


print(f"Accuracy: {accuracy:.2f}")
print(report)

Accuracy: 0.71
              precision    recall  f1-score   support

         0.0       0.82      0.62      0.71       284
         1.0       0.46      0.19      0.27       159
         2.0       0.69      0.94      0.80       442

    accuracy                           0.71       885
   macro avg       0.66      0.59      0.59       885
weighted avg       0.69      0.71      0.67       885

